In [1]:
import sys; sys.path.append('../..')
import pandas as pd
from src.modelos.dados import ler_base, preparar

base = preparar(ler_base())
ano_def = pd.to_datetime(base['data_default']).dt.year
d = base[(base['default'] == 1) & (ano_def <= 2017)].copy()

recuperado = d['recuperacao_principal'].fillna(0) + d['recuperacao_juros'].fillna(0)
d['lgd'] = ((d['ead1'] - recuperado) / d['ead1']).clip(0, 1)
d = d[d['ead1'] > 0]

print('Defaults usados:', len(d))
print('LGD média:', round(d['lgd'].mean(), 3), '| mediana:', round(d['lgd'].median(), 3))
print('\nPor país:\n', d.groupby('pais')['lgd'].agg(['size', 'mean']).round(3))
print('\nLGD do Bondora (média):', round(d['lgd_bondora'].mean(), 3))

Defaults usados: 13755
LGD média: 0.599 | mediana: 0.756

Por país:
       size   mean
pais             
EE    5745  0.443
ES    4664  0.790
FI    3346  0.600

LGD do Bondora (média): 0.752


In [3]:
d['ano_def'] = pd.to_datetime(d['data_default']).dt.year
print(d.pivot_table(index='ano_def', columns='pais', values='lgd', aggfunc='mean').round(3))
print(d.pivot_table(index='ano_def', columns='pais', values='lgd', aggfunc='size'))

pais        EE     ES     FI
ano_def                     
2009     0.377    NaN    NaN
2010     0.256    NaN    NaN
2011     0.088    NaN    NaN
2012     0.120    NaN    NaN
2013     0.203    NaN  0.081
2014     0.326  0.802  0.451
2015     0.346  0.743  0.565
2016     0.488  0.799  0.606
2017     0.612  0.821  0.651
pais         EE      ES      FI
ano_def                        
2009       79.0     NaN     NaN
2010      293.0     NaN     NaN
2011      224.0     NaN     NaN
2012      124.0     NaN     NaN
2013      124.0     NaN     2.0
2014      485.0   345.0   163.0
2015     1099.0  1324.0   956.0
2016     1509.0  1646.0  1201.0
2017     1808.0  1349.0  1024.0


In [4]:
ok = d['ano_def'].between(2014, 2015)
print(d[ok].groupby('pais')['lgd'].agg(['size', 'mean']).round(3))

      size   mean
pais             
EE    1584  0.340
ES    1669  0.755
FI    1119  0.549
